# DR9 Outer-Radius Spearman Sweep

This notebook measures local DR9 photometric-galaxy environments around redMaPPer clusters using cumulative projected annuli

\[
R_{\rm min} < R < R_{\rm out},
\]

with fixed inner radius \(R_{\rm min}=1.5\ h^{-1}{\rm Mpc}\) and a grid of outer radii \(R_{\rm out}=\{5,6,7,8,9,10\}\ h^{-1}{\rm Mpc}\).

The important implementation detail is that the DR9 sweep files are read only once. For each sweep file, the notebook computes all outer-radius definitions while that file is in memory, accumulates the counts, and then deletes the file before moving on.

In [ ]:
from pathlib import Path
import gc
import pickle
import sys
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, unique
from scipy.stats import binned_statistic, pearsonr, rankdata, spearmanr

# Make local_overdensity helper functions importable whether this notebook is run
# from the repo root or from inside local_overdensity.
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == 'local_overdensity':
    LOCAL_OVERDENSITY_DIR = NOTEBOOK_DIR
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR
    LOCAL_OVERDENSITY_DIR = REPO_ROOT / 'local_overdensity'

if str(LOCAL_OVERDENSITY_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_OVERDENSITY_DIR))

from DR9_localOverdensity_sweep_mpi import *

# The helper script is batch-oriented; switch plotting back to inline for notebooks.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

In [ ]:
# Configuration: edit these values if needed, then run the notebook top to bottom.

INPUT_DIR = REPO_ROOT / 'catalogs'
OUTPUT_DIR = REPO_ROOT / 'local_overdensity' / 'dr9_outputs'
RADIUS_SWEEP_OUTPUT_DIR = OUTPUT_DIR / 'outer_radius_spearman_sweep_notebook'
RADIUS_SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATALOG_PATH = INPUT_DIR / 'bgs_clus_RM_gal_matched.pickle'

SWEEP_DIRS = [
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/north/sweep/9.0'),
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/south/sweep/9.0'),
]
SWEEP_PATTERN = 'sweep-*.fits'
MAX_SWEEP_FILES = None   # set to e.g. 5 for a quick test

RMIN_HMPC = 1.5
ROUT_HMPC_GRID = np.array([5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
R_MAG_LIMIT = 22.0

NSIDE_GAL = 4096
NSIDE_SWEEP_COVERAGE = 1024
NBINS = 8

PRIMARY_RICHNESS_COL = 'lambda_spec_tot'
PRIMARY_ENV_COL = 'Sigma_env_covcorr'

print('Repo root:', REPO_ROOT)
print('Catalog:', CATALOG_PATH)
print('Output dir:', RADIUS_SWEEP_OUTPUT_DIR)
print('Cumulative annuli:')
for rout in ROUT_HMPC_GRID:
    print(f'  {RMIN_HMPC:g} < R < {rout:g} h^-1 Mpc')

## Helper Functions

These helpers find sweep files, convert table columns to clean NumPy arrays, compute correlations, and make the binned-mean/rank-rank plots.

In [ ]:
def find_sweep_files(sweep_dirs, pattern='sweep-*.fits', max_files=None):
    files = []
    for sweep_dir in sweep_dirs:
        sweep_dir = Path(sweep_dir)
        files.extend(glob(str(sweep_dir / pattern)))
        files.extend(glob(str(sweep_dir / '*' / pattern)))
    files = sorted(set(files))
    if max_files is not None:
        files = files[:max_files]
    if len(files) == 0:
        raise FileNotFoundError(
            f'No sweep files matched {[str(Path(d) / pattern) for d in sweep_dirs]}'
        )
    return [Path(f) for f in files]


def col_float(table, col):
    arr = np.ma.asarray(table[col], dtype=float)
    return np.ma.filled(arr, np.nan)


def finite_positive(x):
    x = np.asarray(x, dtype=float)
    return np.isfinite(x) & (x > 0)


def correlation_summary(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = finite_positive(x) & np.isfinite(y)
    if np.count_nonzero(mask) < 4:
        return dict(
            N=int(np.count_nonzero(mask)),
            spearman_r=np.nan,
            spearman_p=np.nan,
            pearson_r_logx=np.nan,
            pearson_p_logx=np.nan,
        )

    spearman_r, spearman_p = spearmanr(x[mask], y[mask])
    pearson_r_logx, pearson_p_logx = pearsonr(np.log10(x[mask]), y[mask])
    return dict(
        N=int(np.count_nonzero(mask)),
        spearman_r=float(spearman_r),
        spearman_p=float(spearman_p),
        pearson_r_logx=float(pearson_r_logx),
        pearson_p_logx=float(pearson_p_logx),
    )


def add_binned_mean(ax, x, y, nbins=8, color='crimson', label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = finite_positive(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return

    bins = np.logspace(np.log10(np.nanmin(x)), np.log10(np.nanmax(x)), nbins + 1)
    centers = np.sqrt(bins[:-1] * bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    good = count > 3
    sem = std / np.sqrt(np.clip(count, 1, None))

    ax.errorbar(
        centers[good],
        mean[good],
        yerr=sem[good],
        fmt='o',
        color=color,
        ecolor=color,
        capsize=3,
        label=label,
        zorder=5,
    )


def add_binned_mean_linear_x(ax, x, y, nbins=8, color='crimson', label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return

    bins = np.linspace(np.nanmin(x), np.nanmax(x), nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    good = count > 3
    sem = std / np.sqrt(np.clip(count, 1, None))

    ax.errorbar(
        centers[good],
        mean[good],
        yerr=sem[good],
        fmt='o',
        color=color,
        ecolor=color,
        capsize=3,
        label=label,
        zorder=5,
    )

## Load Cluster Catalog And Sweep File List

In [ ]:
with CATALOG_PATH.open('rb') as handle:
    bgs_matched = pickle.load(handle)

rm_tab = unique(bgs_matched, keys='ID')

ra_cl = np.asarray(rm_tab['RA_x'], dtype=float)
dec_cl = np.asarray(rm_tab['DEC_x'], dtype=float)
z_cl = np.asarray(rm_tab['Z_SPEC_x'], dtype=float)
n_cl = len(rm_tab)

sweep_files = find_sweep_files(SWEEP_DIRS, SWEEP_PATTERN, max_files=MAX_SWEEP_FILES)

print(f'Clusters: {n_cl:,}')
print(f'Sweep files: {len(sweep_files):,}')
print('Richness-like columns present:')
for col in ['lambda_spec_tot', 'lambda_spec', 'lambda_true', 'LAMBDA']:
    if col in rm_tab.colnames:
        print(' ', col)

## Precompute Annulus Areas

The geometric area depends on cluster redshift because a fixed comoving projected radius corresponds to a redshift-dependent angular radius.

In [ ]:
n_r = len(ROUT_HMPC_GRID)
area_env_deg2_all = np.zeros((n_r, n_cl), dtype=float)

for ir, rout in enumerate(ROUT_HMPC_GRID):
    _, area_env_deg2_all[ir] = annulus_area_for_clusters(z_cl, RMIN_HMPC, rout)

print('Area array shape:', area_env_deg2_all.shape)

## One-Pass DR9 Sweep Loop

This is the core step. Each sweep file is read once. While that file is in memory, the notebook computes all cumulative annuli \(1.5<R<X\) and accumulates counts/covered areas for every \(X\).

In [ ]:
n_env_all_radii = np.zeros((n_r, n_cl), dtype=np.int64)
covered_area_env_all_radii = np.zeros((n_r, n_cl), dtype=float)
n_files_touching_cluster = np.zeros(n_cl, dtype=np.int64)

for i_file, sweep_file in enumerate(sweep_files, start=1):
    print(f'[{i_file:05d}/{len(sweep_files):05d}] {sweep_file.name}', flush=True)

    sweep_bounds = read_sweep_bounds(sweep_file)
    overlap = clusters_overlapping_sweep_box(
        ra_cl,
        dec_cl,
        z_cl,
        sweep_bounds,
        rmax_hmpc=float(np.max(ROUT_HMPC_GRID)),
    )
    idx = np.where(overlap)[0]
    if len(idx) == 0:
        continue

    dr9 = read_one_dr9_sweep(sweep_file, r_mag_limit=R_MAG_LIMIT)
    ra_gal = np.asarray(dr9['RA'], dtype=float)
    dec_gal = np.asarray(dr9['DEC'], dtype=float)

    n_files_touching_cluster[idx] += 1

    for ir, rout in enumerate(ROUT_HMPC_GRID):
        counts, _ = count_galaxies_in_annuli(
            ra_cl[idx],
            dec_cl[idx],
            z_cl[idx],
            ra_gal,
            dec_gal,
            rmin_hmpc=RMIN_HMPC,
            rmax_hmpc=float(rout),
            nside=NSIDE_GAL,
        )
        coverage, _ = sweep_annulus_coverage_for_clusters(
            ra_cl[idx],
            dec_cl[idx],
            z_cl[idx],
            sweep_bounds,
            rmin_hmpc=RMIN_HMPC,
            rmax_hmpc=float(rout),
            nside=NSIDE_SWEEP_COVERAGE,
        )
        n_env_all_radii[ir, idx] += counts
        covered_area_env_all_radii[ir, idx] += (
            np.nan_to_num(coverage, nan=0.0) * area_env_deg2_all[ir, idx]
        )

    del dr9, ra_gal, dec_gal, counts, coverage
    gc.collect()

print('Finished one-pass sweep.')

## Build Radius Tables And Correlation Summary

For each outer radius, this creates a cluster table with the count, geometric surface density, coverage-corrected surface density, and coverage diagnostic. Then it computes Spearman and Pearson correlations against the available richness columns.

In [ ]:
richness_cols = [
    col for col in ['lambda_spec_tot', 'lambda_spec', 'lambda_true', 'LAMBDA']
    if col in rm_tab.colnames
]
if PRIMARY_RICHNESS_COL not in richness_cols:
    raise KeyError(f'{PRIMARY_RICHNESS_COL} is missing. Available columns: {richness_cols}')

summary_rows = []
radius_tables = {}

for ir, rout in enumerate(ROUT_HMPC_GRID):
    coverage_env = np.clip(
        safe_divide(covered_area_env_all_radii[ir], area_env_deg2_all[ir]),
        0.0,
        1.0,
    )
    sigma_env_geom = safe_divide(n_env_all_radii[ir], area_env_deg2_all[ir])
    sigma_env_covcorr = safe_divide(n_env_all_radii[ir], covered_area_env_all_radii[ir])

    tab = rm_tab.copy()
    tab['Rmin_hmpc'] = np.full(n_cl, RMIN_HMPC)
    tab['Rout_hmpc'] = np.full(n_cl, rout)
    tab['N_env'] = n_env_all_radii[ir]
    tab['area_env_deg2'] = area_env_deg2_all[ir]
    tab['covered_area_env_deg2'] = covered_area_env_all_radii[ir]
    tab['coverage_env_sweep'] = coverage_env
    tab['n_files_touching_cluster'] = n_files_touching_cluster
    tab['Sigma_env_geom'] = sigma_env_geom
    tab['Sigma_env_covcorr'] = sigma_env_covcorr

    suffix = f'R{RMIN_HMPC:g}_{rout:g}'.replace('.', 'p')
    tab.write(
        RADIUS_SWEEP_OUTPUT_DIR / f'rm_dr9_environment_outer_radius_{suffix}.ecsv',
        format='ascii.ecsv',
        overwrite=True,
    )
    tab.write(
        RADIUS_SWEEP_OUTPUT_DIR / f'rm_dr9_environment_outer_radius_{suffix}.fits',
        overwrite=True,
    )
    radius_tables[float(rout)] = tab

    y_arrays = {
        'N_env': np.asarray(tab['N_env'], dtype=float),
        'Sigma_env_geom': np.asarray(tab['Sigma_env_geom'], dtype=float),
        'Sigma_env_covcorr': np.asarray(tab['Sigma_env_covcorr'], dtype=float),
    }

    for richness_col in richness_cols:
        x = col_float(tab, richness_col)
        for y_col, y in y_arrays.items():
            stats = correlation_summary(x, y)
            summary_rows.append({
                'rmin_hmpc': RMIN_HMPC,
                'rout_hmpc': float(rout),
                'richness_col': richness_col,
                'y_col': y_col,
                **stats,
            })

summary = Table(rows=summary_rows)
summary['abs_spearman_r'] = np.abs(np.asarray(summary['spearman_r'], dtype=float))
summary.sort('abs_spearman_r')
summary.reverse()
summary.write(
    RADIUS_SWEEP_OUTPUT_DIR / 'outer_radius_spearman_summary.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)

summary[:12]

## Best Radius For The Fiducial Metric

By default, the fiducial comparison is

\[
\rho_S\left(\lambda_{\rm spec,tot}, \Sigma_{\rm env,covcorr}\right),
\]

and the best radius is selected by maximizing \(|\rho_S|\).

In [ ]:
primary = summary[
    (np.asarray(summary['richness_col']) == PRIMARY_RICHNESS_COL)
    & (np.asarray(summary['y_col']) == PRIMARY_ENV_COL)
]

best_idx = int(np.nanargmax(np.abs(primary['spearman_r'])))
best_rout = float(primary['rout_hmpc'][best_idx])
best_spearman = float(primary['spearman_r'][best_idx])
best_p = float(primary['spearman_p'][best_idx])

print(f'Best radius for {PRIMARY_ENV_COL} vs {PRIMARY_RICHNESS_COL}:')
print(f'  {RMIN_HMPC:g} < R < {best_rout:g} h^-1 Mpc')
print(f'  Spearman r_s = {best_spearman:.4f}, p = {best_p:.3e}')

primary

## Correlation Coefficient Versus Outer Radius

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.axhline(0.0, color='black', lw=1.0, alpha=0.5)
ax.plot(
    primary['rout_hmpc'],
    primary['spearman_r'],
    marker='o',
    color='crimson',
    label='Spearman',
)
ax.plot(
    primary['rout_hmpc'],
    primary['pearson_r_logx'],
    marker='s',
    color='royalblue',
    label=r'Pearson $(\log x,y)$',
)
ax.scatter(
    best_rout,
    best_spearman,
    s=100,
    facecolor='none',
    edgecolor='black',
    lw=1.6,
    zorder=5,
    label='max |Spearman|',
)
ax.set_xlabel(r'$R_{\rm out}\ [h^{-1}{\rm Mpc}]$')
ax.set_ylabel('correlation coefficient')
ax.set_title(f'{PRIMARY_ENV_COL} versus {PRIMARY_RICHNESS_COL}')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(
    RADIUS_SWEEP_OUTPUT_DIR / f'radius_sweep_correlations_{PRIMARY_RICHNESS_COL}_{PRIMARY_ENV_COL}.png',
    dpi=180,
)
plt.show()

## Binned Mean And Rank-Rank Plot For The Best Radius

The left panel shows the raw relation with binned means and SEM error bars. The right panel shows the same data in rank space, which is the space Spearman correlation actually uses.

In [ ]:
best_table = radius_tables[best_rout]
x = col_float(best_table, PRIMARY_RICHNESS_COL)
y = col_float(best_table, PRIMARY_ENV_COL)
mask = finite_positive(x) & np.isfinite(y)
x = x[mask]
y = y[mask]

sr, sp = spearmanr(x, y)
pr, pp = pearsonr(np.log10(x), y)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

axes[0].scatter(x, y, s=12, alpha=0.35, color='0.25', edgecolor='none')
add_binned_mean(axes[0], x, y, nbins=NBINS, color='crimson')
axes[0].set_xscale('log')
axes[0].set_xlabel(PRIMARY_RICHNESS_COL)
axes[0].set_ylabel(PRIMARY_ENV_COL)
axes[0].set_title(rf'${RMIN_HMPC:g}<R<{best_rout:g}\ h^{{-1}}\,{{\rm Mpc}}$')
axes[0].text(
    0.04,
    0.96,
    rf'Spearman $r_s={sr:.3f}$, $p={sp:.1e}$'
    + '\n'
    + rf'Pearson $(\log x,y)$ $r={pr:.3f}$, $p={pp:.1e}$',
    transform=axes[0].transAxes,
    ha='left',
    va='top',
    fontsize=10,
    bbox=dict(facecolor='white', edgecolor='none', alpha=0.8),
)
axes[0].legend(frameon=False)

rx = rankdata(x)
ry = rankdata(y)
axes[1].scatter(rx, ry, s=12, alpha=0.35, color='0.25', edgecolor='none')
add_binned_mean_linear_x(axes[1], rx, ry, nbins=NBINS, color='crimson')
axes[1].plot(
    [np.nanmin(rx), np.nanmax(rx)],
    [np.nanmin(rx), np.nanmax(rx)],
    color='black',
    lw=1.0,
    alpha=0.35,
)
axes[1].set_xlabel(f'rank({PRIMARY_RICHNESS_COL})')
axes[1].set_ylabel(f'rank({PRIMARY_ENV_COL})')
axes[1].set_title('Rank-rank visualization')

fig.tight_layout()
fig.savefig(
    RADIUS_SWEEP_OUTPUT_DIR / f'best_radius_{PRIMARY_RICHNESS_COL}_{PRIMARY_ENV_COL}_binned_mean_rank_rank.png',
    dpi=180,
)
plt.show()

## Optional: Compare RedMaPPer Richness Directly

This cell repeats the radius-dependence plot for \(\lambda_{\rm RM}\), if the `LAMBDA` column exists.

In [ ]:
if 'LAMBDA' in richness_cols:
    rm_summary = summary[
        (np.asarray(summary['richness_col']) == 'LAMBDA')
        & (np.asarray(summary['y_col']) == PRIMARY_ENV_COL)
    ]

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    ax.axhline(0.0, color='black', lw=1.0, alpha=0.5)
    ax.plot(
        rm_summary['rout_hmpc'],
        rm_summary['spearman_r'],
        marker='o',
        color='darkorange',
        label='Spearman',
    )
    ax.plot(
        rm_summary['rout_hmpc'],
        rm_summary['pearson_r_logx'],
        marker='s',
        color='royalblue',
        label=r'Pearson $(\log x,y)$',
    )
    ax.set_xlabel(r'$R_{\rm out}\ [h^{-1}{\rm Mpc}]$')
    ax.set_ylabel('correlation coefficient')
    ax.set_title(f'{PRIMARY_ENV_COL} versus LAMBDA')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        RADIUS_SWEEP_OUTPUT_DIR / f'radius_sweep_correlations_LAMBDA_{PRIMARY_ENV_COL}.png',
        dpi=180,
    )
    plt.show()
else:
    print('No LAMBDA column found.')

## Saved Outputs

The notebook writes one table per tested radius, a summary table, and the diagnostic figures.

In [ ]:
print('Output directory:', RADIUS_SWEEP_OUTPUT_DIR)
print('Summary table:', RADIUS_SWEEP_OUTPUT_DIR / 'outer_radius_spearman_summary.ecsv')
print('Files written:')
for path in sorted(RADIUS_SWEEP_OUTPUT_DIR.glob('*')):
    print(' ', path.name)